<a href="https://colab.research.google.com/github/avithegr8-coder/Hackathon-SUDATA-2026/blob/main/Hackathon_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ____________________________________________________________
# NSW LOST PROPERTY REPORTING SYSTEM
# Hackathon 2026 — Team Project
# ────────────────────────────────────────────────────────────
# OVERVIEW:
# Helps passengers report lost items on NSW transport.
# Collects their journey details, matches the correct operator
# from NSW routes dataset, and generates a structured
# reference number for follow-up.
# ────────────────────────────────────────────────────────────

# ── DEPENDENCIES ────────────────────────────────────────────
import pandas as pd
from datetime import datetime


# ____________________________________________________________
# PART 1 — LOAD DATA
# ────────────────────────────────────────────────────────────
# Load the NSW routes_1 CSV into the files and prepare for operator matching.
# Core dataset that tells us which operator
# runs which route and their contact details.
# ────────────────────────────────────────────────────────────

df = pd.read_csv('routes_1.csv')
def load_data():
    df = pd.read_csv('routes_1.csv')
    return df


# ____________________________________________________________
# PART 2 — COLLECT USER INPUT
# ────────────────────────────────────────────────────────────
# Ask the passenger for details about lost item and journey.
# Used to match the operator and generate outputs.
# Info: Name, Location, DateTime, Item Desc, Contacts Details
# ────────────────────────────────────────────────────────────

def get_user_details():
    print("--- NSW LOST PROPERTY REPORTING ---")
    name = input("Name: ")
    item = input("What did you lose? (e.g. Black Wallet): ")
    date_str = input("Date lost (DD/MM/YYYY): ")
    date_dt = datetime.strptime(date_str, "%d/%m/%Y")
    time_str = input("Approximate time (e.g. 8:30am): ").upper()
    time_dt = datetime.strptime(time_str, "%I:%M%p")

    print("\n--- JOURNEY DETAILS ---")
    mode = input("Transport Mode (Bus, Ferry, Train & Metro): ").strip().lower()
    keyword = input("Route Number (T1, 350): ").strip().lower()

    return {"name": name,
            "item": item,
            "date": date_dt,
            "time": time_dt,
            "mode": mode,
            "keyword": keyword}


# ____________________________________________________________
# PART 3 — MATCH OPERATOR FROM CSV
# ────────────────────────────────────────────────────────────
# Search the routes dataset using the passenger's transport type
# and suburb/route keyword. Return the operator name and
# their contact details.
# ────────────────────────────────────────────────────────────

def find_operator(df, mode, keyword):

    # Map user input to CSV values
    mode_map = {'bus': 'Bus', 'train': 'Commuter railway', 'metro': 'Subway', 'ferry': 'Ferry', 'tram': 'Tram', 'coach': 'Coach'}
    mapped_mode = mode_map.get(mode.lower(), None)
    if mapped_mode is None:
        print(f"⚠️  Transport mode '{mode}' not recognised.")
        return "Transport NSW", "131 500"

    # Filter by transport mode or route number
    results = df[df['mot_for_interchange'] == mapped_mode]
    results = results[
    results['my_timetable_route_name'].str.contains(keyword, case=False, na=False)]
    # Return first match
    if results.empty:
        print(f"⚠️  No operator found for '{keyword}'. Defaulting to Transport NSW.")
        return "Transport NSW", "131 500"
    match = results.iloc[0]
    phone = match['lp_contact_number']
    if pd.isna(phone): phone = "131 500"
    return match['operator_name'], phone


# ____________________________________________________________
# PART 4 — OUTPUT REF. & DETAILS SENT TO SERVICE FOR NOTICE
# ────────────────────────────────────────────────────────────
# Build a structured reference number from the incident details.
# Display a clean output card the passenger can screenshot
# and use for follow-up contact.
#
# Reference format: LP-YYYYMMDD-MODE-XXXX
# Example:          LP-20260418-BUS-0042
#
# Send contact details if users want to follow up immediately.
# ────────────────────────────────────────────────────────────

counter = 1
def g_d_report(user_data, operator_name, operator_phone):
    global counter
    date_ref = user_data['date'].strftime("%Y%m%d")
    mode_ref = user_data['mode'][:3].upper()
    order_id = f"{counter:04d}"
    ref_num = f"LP-{date_ref}-{mode_ref}-{order_id}"
    counter += 1
    if counter > 9999: counter = 1

    print("\n" + "═"*55)
    print("       NSW TRANSPORT LOST PROPERTY OFFICIAL REPORT       ")
    print("═"*55)

    print(f" REFERENCE: {ref_num}")
    print(f" PASSENGER: {user_data['name'].upper()}")
    print("-" * 55)
    lost_on = user_data['date'].strftime("%d %B %Y")
    lost_at = user_data['time'].strftime("%I:%M %p")

    print(f" ITEM:      {user_data['item']}")
    print(f" LOST ON:   {lost_on} approx. {lost_at}")
    print(f" ROUTE/REF: {user_data['keyword'].upper()} ({user_data['mode'].capitalize()})")
    print("-" * 55)

    print(" RECOMMENDED CONTACT FOR FOLLOW-UP:")
    print(f" OPERATOR:  {operator_name}")
    print(f" PHONE:     {operator_phone}")

    print("═"*55)
    print(" Please screenshot this card for your records.")
    print(" Quote your Reference Number when contacting the operator for followups.")
    print(" This record will be sent to the service centre, which would call you within 2 business hours depending on workload.")
    print("═"*55 + "\n")


# ____________________________________________________________
# MAIN — RUN THE PROGRAM
# ────────────────────────────────────────────────────────────

df = load_data()
user_data = get_user_details()
operator_name, operator_phone = find_operator(df, user_data['mode'], user_data['keyword'])
g_d_report(user_data, operator_name, operator_phone)

FileNotFoundError: [Errno 2] No such file or directory: 'routes_1.csv'